In [3]:
import numpy as np
import pandas as pd

# --------------------------------------------------
# Model weights (given)
# --------------------------------------------------
weights = {
    "intercept": -3.82398,
    "age": -0.02990,
    "band_b": -0.09089,
    "band_c": -0.19558,
    "shop_value": 0.02999,
    "shop_frequency": 0.74572
}

# --------------------------------------------------
# Logistic (sigmoid) function
# --------------------------------------------------
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# --------------------------------------------------
# One-hot encoding for socioeconomic band
# a -> b=0, c=0
# b -> b=1, c=0
# c -> b=0, c=1
# --------------------------------------------------
def encode_band(band):
    if band.lower() == "b":
        return 1, 0
    elif band.lower() == "c":
        return 0, 1
    else:  # band "a"
        return 0, 0

# --------------------------------------------------
# Prediction function
# --------------------------------------------------
def predict_probability(age, band, shop_freq, shop_value):
    b, c = encode_band(band)

    z = (
        weights["intercept"]
        + weights["age"] * age
        + weights["band_b"] * b
        + weights["band_c"] * c
        + weights["shop_value"] * shop_value
        + weights["shop_frequency"] * shop_freq
    )

    return z, sigmoid(z)

# --------------------------------------------------
# Input data (query instances)
# --------------------------------------------------
data = pd.DataFrame({
    "ID": [1, 2, 3, 4, 5],
    "AGE": [56, 21, 48, 37, 32],
    "BAND": ["b", "c", "b", "c", "a"],
    "SHOP_FREQUENCY": [1.60, 4.92, 1.21, 0.72, 1.08],
    "SHOP_VALUE": [109.32, 11.28, 161.19, 170.65, 165.39]
})

# --------------------------------------------------
# Run predictions
# --------------------------------------------------
results = []

for _, row in data.iterrows():
    z, p = predict_probability(
        age=row["AGE"],
        band=row["BAND"],
        shop_freq=row["SHOP_FREQUENCY"],
        shop_value=row["SHOP_VALUE"]
    )

    results.append({
        "ID": row["ID"],
        "Linear score (z)": round(z, 4),
        "Probability": round(p, 4)
    })

results_df = pd.DataFrame(results)
print(results_df)


   ID  Linear score (z)  Probability
0   1           -1.1176       0.2465
1   2           -0.6402       0.3452
2   3            0.3863       0.5954
3   4            0.5289       0.6292
4   5            0.9846       0.7280


In [2]:
import numpy as np
import pandas as pd

# --------------------------------------------------
# Summary statistics from data quality report
# --------------------------------------------------
STATS = {
    "AGE": {"min": 18, "max": 63, "mean": 32.7},
    "SHOP_FREQUENCY": {"min": 0.2, "max": 5.4, "mean": 2.2},
    "SHOP_VALUE": {"min": 5, "max": 230.7, "mean": 101.9}
}

MODE_BAND = "a"

# --------------------------------------------------
# Logistic regression weights (after normalization)
# --------------------------------------------------
weights = {
    "intercept": 0.6679,
    "age": -0.5795,
    "band_b": -0.1981,
    "band_c": -0.2318,
    "shop_value": 3.4091,
    "shop_frequency": 2.0499
}

def range_normalize(x, xmin, xmax):
    return 2 * (x - xmin) / (xmax - xmin) - 1

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def encode_band(band):
    band = str(band).lower()
    if band == "b":
        return 1, 0
    elif band == "c":
        return 0, 1
    else:  # baseline a (or anything imputed to a)
        return 0, 0

# --------------------------------------------------
# Query instances (np.nan for missing is fine)
# --------------------------------------------------
data = pd.DataFrame({
    "ID": [1, 2, 3, 4, 5],
    "AGE": [38, 56, 18, np.nan, 62],
    "BAND": ["a", "b", "c", "b", np.nan],
    "SHOP_FREQUENCY": [1.90, 1.60, 6.00, 1.33, 0.85],
    "SHOP_VALUE": [165.39, 109.32, 10.09, 204.62, 110.50]
})

# Toggle this to match Excel assumption
CLIP_CONTINUOUS_TO_RANGE = True

results = []

for _, row in data.iterrows():

    # --- Impute missing values (use pd.isna for NaN detection) ---
    age = STATS["AGE"]["mean"] if pd.isna(row["AGE"]) else float(row["AGE"])
    band = MODE_BAND if pd.isna(row["BAND"]) else row["BAND"]

    freq = float(row["SHOP_FREQUENCY"])
    value = float(row["SHOP_VALUE"])

    # --- Optional clipping to [min, max] BEFORE normalization ---
    if CLIP_CONTINUOUS_TO_RANGE:
        age = np.clip(age, STATS["AGE"]["min"], STATS["AGE"]["max"])
        freq = np.clip(freq, STATS["SHOP_FREQUENCY"]["min"], STATS["SHOP_FREQUENCY"]["max"])
        value = np.clip(value, STATS["SHOP_VALUE"]["min"], STATS["SHOP_VALUE"]["max"])

    # --- Normalize ---
    age_n = range_normalize(age, STATS["AGE"]["min"], STATS["AGE"]["max"])
    freq_n = range_normalize(freq, STATS["SHOP_FREQUENCY"]["min"], STATS["SHOP_FREQUENCY"]["max"])
    value_n = range_normalize(value, STATS["SHOP_VALUE"]["min"], STATS["SHOP_VALUE"]["max"])

    # --- Encode categorical feature ---
    b, c = encode_band(band)

    # --- Linear score ---
    z = (
        weights["intercept"]
        + weights["age"] * age_n
        + weights["band_b"] * b
        + weights["band_c"] * c
        + weights["shop_value"] * value_n
        + weights["shop_frequency"] * freq_n
    )

    p = sigmoid(z)

    results.append({
        "ID": int(row["ID"]),
        "z": z,
        "Probability": p
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))


 ID         z  Probability
  1  1.458850     0.811357
  2 -1.133203     0.243571
  3 -0.189836     0.452683
  4  2.132957     0.894065
  5 -1.645307     0.161744


In [26]:
import numpy as np

# -------------------------
# Sigmoid
# -------------------------
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# -------------------------
# Loss (0/1 version)
# -------------------------
def loss_01(w, x, y):
    z = np.dot(w, x)
    s = sigmoid(z)
    return -y*np.log(s) - (1-y)*np.log(1-s)

# -------------------------
# Analytical gradient
# -------------------------
def grad_01(w, x, y):
    z = np.dot(w, x)
    s = sigmoid(z)
    return x * (s - y)

# -------------------------
# Numerical gradient
# -------------------------
def numerical_grad(f, w, eps=1e-6):
    grad = np.zeros_like(w)
    for i in range(len(w)):
        w_plus = w.copy()
        w_minus = w.copy()
        w_plus[i] += eps
        w_minus[i] -= eps
        grad[i] = (f(w_plus) - f(w_minus)) / (2*eps)
    return grad

# Test
x = np.array([1.0, 2.0, -1.5])
w = np.array([0.3, -0.4, 0.2])
y = 1 # either 0 or 1

analytical = grad_01(w, x, y)
numerical = numerical_grad(lambda w_: loss_01(w_, x, y), w)

print("Analytical:", analytical)
print("Numerical :", numerical)


Analytical: [-0.68997448 -1.37994896  1.03496172]
Numerical : [-0.68997448 -1.37994896  1.03496172]


In [28]:
# -------------------------
# Loss (-1/+1 version)
# -------------------------
def loss_pm1(w, x, y):
    z = np.dot(w, x)
    return np.log(1 + np.exp(-y * z))

# -------------------------
# Analytical gradient
# -------------------------
def grad_pm1(w, x, y):
    z = np.dot(w, x)
    return -y * x * (1 / (1 + np.exp(y * z)))

# Test
x = np.array([1.0, 2.0, -1.5])
w = np.array([0.3, -0.4, 0.2])
y = 1  # try +1 or -1

analytical = grad_pm1(w, x, y)
numerical = numerical_grad(lambda w_: loss_pm1(w_, x, y), w)

print("Analytical:", analytical)
print("Numerical :", numerical)


Analytical: [-0.68997448 -1.37994896  1.03496172]
Numerical : [-0.68997448 -1.37994896  1.03496172]
